# Qwen2.5-0.5B LunarLander SFT with LLaMA-Factory

This notebook follows the official **LLaMA-Factory supervised fine-tuning (SFT)** flow for the dataset `Ali2023kosemen/lunar_lander_270_reward`.

The notebook is organized as:

1. **Installation**
2. **Data Preparation**
3. **Training with `torchrun src/train.py`**
4. **Training log inspection**
5. **Post-training inference tests**
6. **Optional model archiving and download**

This notebook is tuned conservatively for **Colab T4 16 GB** and uses **full fine-tuning** (`--finetuning_type full`) instead of LoRA.

In [ ]:
import platform
import torch

print("Platform:", platform.platform())
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Colab, go to Runtime -> Change runtime type -> T4 GPU, then rerun this cell."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
print("GPU:", gpu_name)
print("GPU memory (GB):", gpu_mem_gb)

if gpu_mem_gb < 14:
    raise RuntimeError(
        f"Detected only {gpu_mem_gb} GB VRAM. This notebook is tuned for Colab T4 16 GB or better."
    )

## 1. Installation

This section prepares a clean Colab environment and installs LLaMA-Factory.

Notes for Colab T4:
- `--deepspeed` is not used to keep the single-GPU run simpler.
- `--flash_attn` is not used to avoid extra installation and compatibility issues.
- `fp16` is used instead of `bf16`.

In [ ]:
%%capture
!pip install -U datasets huggingface_hub pandas matplotlib accelerate
!rm -rf /content/LLaMA-Factory
!git clone https://github.com/hiyouga/LLaMA-Factory.git /content/LLaMA-Factory
%cd /content/LLaMA-Factory
!pip install -e ".[torch,metrics]"

## 2. Data Preparation

The official docs expect a custom dataset in `data/` and a matching `dataset_info.json` entry.

Our Hugging Face dataset uses `human` / `gpt` roles. For better alignment with the ShareGPT example in the docs, we normalize them to:
- `human -> user`
- `gpt -> assistant`

In [ ]:
from datasets import load_dataset
import json
from pathlib import Path

DATASET_ID = "Ali2023kosemen/lunar_lander_270_reward"
DATA_DIR = Path("/content/LLaMA-Factory/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

json_path = DATA_DIR / "lunar_lander_270_reward.json"
info_path = DATA_DIR / "dataset_info.json"

ds = load_dataset(DATASET_ID, split="train")

converted_rows = []
for row in ds:
    converted = {"conversations": []}
    for msg in row["conversations"]:
        role = msg["from"]
        if role == "human":
            role = "user"
        elif role == "gpt":
            role = "assistant"
        converted["conversations"].append({
            "from": role,
            "value": msg["value"],
        })
    if "system" in row and row["system"] is not None:
        converted["system"] = row["system"]
    if "tools" in row and row["tools"] is not None:
        converted["tools"] = row["tools"]
    converted_rows.append(converted)

with json_path.open("w", encoding="utf-8") as f:
    json.dump(converted_rows, f, ensure_ascii=False, indent=2)

if info_path.exists():
    dataset_info = json.loads(info_path.read_text(encoding="utf-8"))
else:
    dataset_info = {}

dataset_info["lunar_lander_270_reward"] = {
    "file_name": "lunar_lander_270_reward.json",
    "formatting": "sharegpt",
    "columns": {
        "messages": "conversations"
    },
    "tags": {
        "role_tag": "from",
        "content_tag": "value",
        "user_tag": "user",
        "assistant_tag": "assistant"
    }
}

if any("system" in row for row in converted_rows):
    dataset_info["lunar_lander_270_reward"]["columns"]["system"] = "system"

if any("tools" in row for row in converted_rows):
    dataset_info["lunar_lander_270_reward"]["columns"]["tools"] = "tools"

info_path.write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved dataset to:", json_path)
print("Updated dataset info:", info_path)
print("Rows:", len(converted_rows))
print("Sample row:")
print(json.dumps(converted_rows[0], ensure_ascii=False, indent=2))

## 3. Training with `torchrun src/train.py`

This is the LLaMA-Factory SFT launch step.

Compared with the documentation example:
- `--finetuning_type full` is used because we want **LoRA-free full fine-tuning**
- `--fp16` is used for Colab T4
- sequence length and effective batch are reduced conservatively to avoid OOM

In [ ]:
from pathlib import Path
import textwrap

OUTPUT_DIR = Path("/content/LLaMA-Factory/saves/qwen25_05b_full_ft_lunarlander_t4")
cutoff_len = 512 if gpu_mem_gb <= 16 else 1024
grad_accum = 32 if gpu_mem_gb <= 16 else 16
warmup_steps = 100 if gpu_mem_gb <= 16 else 50

yaml_text = textwrap.dedent(f"""
### model
model_name_or_path: Qwen/Qwen2.5-0.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: full
gradient_checkpointing: true

### dataset
dataset: lunar_lander_270_reward
template: qwen
cutoff_len: {cutoff_len}
overwrite_cache: true
preprocessing_num_workers: 2
dataloader_num_workers: 2

### output
output_dir: {OUTPUT_DIR.as_posix()}
logging_steps: 10
save_steps: 200
save_total_limit: 2
plot_loss: true
overwrite_output_dir: true
save_only_model: true
report_to: none

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: {grad_accum}
learning_rate: 1.0e-5
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_steps: {warmup_steps}
weight_decay: 0.01
max_grad_norm: 1.0
fp16: true
bf16: false
ddp_timeout: 180000000
""").strip() + "\n"

yaml_path = Path("/content/LLaMA-Factory/qwen25_05b_full_ft_lunarlander_t4.yaml")
yaml_path.write_text(yaml_text, encoding="utf-8")
print("Saved YAML reference to:", yaml_path)
print(yaml_path.read_text())

script_text = textwrap.dedent(f"""
#!/usr/bin/env bash
set -euo pipefail

NPROC_PER_NODE=1
NNODES=1
NODE_RANK=0
MASTER_ADDR=127.0.0.1
MASTER_PORT=29500

DISTRIBUTED_ARGS="
    --nproc_per_node $NPROC_PER_NODE \\
    --nnodes $NNODES \\
    --node_rank $NODE_RANK \\
    --master_addr $MASTER_ADDR \\
    --master_port $MASTER_PORT
"

torchrun $DISTRIBUTED_ARGS src/train.py \\
    --stage sft \\
    --do_train \\
    --use_fast_tokenizer \\
    --model_name_or_path Qwen/Qwen2.5-0.5B-Instruct \\
    --trust_remote_code \\
    --dataset lunar_lander_270_reward \\
    --template qwen \\
    --finetuning_type full \\
    --output_dir {OUTPUT_DIR.as_posix()} \\
    --overwrite_cache \\
    --overwrite_output_dir \\
    --preprocessing_num_workers 2 \\
    --dataloader_num_workers 2 \\
    --warmup_steps {warmup_steps} \\
    --weight_decay 0.01 \\
    --per_device_train_batch_size 1 \\
    --gradient_accumulation_steps {grad_accum} \\
    --ddp_timeout 180000000 \\
    --learning_rate 1e-5 \\
    --lr_scheduler_type cosine \\
    --logging_steps 10 \\
    --cutoff_len {cutoff_len} \\
    --save_steps 200 \\
    --save_total_limit 2 \\
    --plot_loss \\
    --num_train_epochs 1 \\
    --save_only_model \\
    --fp16 \\
    --report_to none
""").strip() + "\n"

script_path = Path("/content/LLaMA-Factory/run_full_ft_t4_torchrun.sh")
script_path.write_text(script_text, encoding="utf-8")
script_path.chmod(0o755)
print("Saved torchrun script to:", script_path)
print(script_path.read_text())

In [ ]:
%cd /content/LLaMA-Factory
!nvidia-smi
!python - <<'PY'
import torch
print('torch.cuda.is_available():', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available in this Colab runtime.')
print('torch.cuda.current_device():', torch.cuda.current_device())
print('torch.cuda.get_device_name(0):', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
PY
!bash /content/LLaMA-Factory/run_full_ft_t4_torchrun.sh

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

log_path = OUTPUT_DIR / 'trainer_log.jsonl'

def resolve_model_dir(output_dir: Path) -> Path:
    checkpoints = sorted(
        [p for p in output_dir.glob('checkpoint-*') if p.is_dir()],
        key=lambda p: int(p.name.split('-')[-1]),
    )
    if checkpoints:
        return checkpoints[-1]
    candidate_files = ['model.safetensors', 'pytorch_model.bin', 'config.json']
    if output_dir.exists() and any((output_dir / name).exists() for name in candidate_files):
        return output_dir
    raise FileNotFoundError(f'No saved model files found under {output_dir}')

if log_path.exists():
    rows = [json.loads(line) for line in log_path.open()]
    hist = pd.DataFrame(rows)
    display(hist.tail())
    if 'loss' in hist.columns:
        train_hist = hist.dropna(subset=['loss'])[['current_steps', 'loss']].copy()
        plt.figure(figsize=(10, 5))
        plt.plot(train_hist['current_steps'], train_hist['loss'], marker='o')
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.title('LLaMA-Factory Full FT Loss')
        plt.grid(alpha=0.3)
        plt.show()
else:
    print('trainer_log.jsonl not found yet. Check the output directory after training finishes.')

MODEL_DIR = resolve_model_dir(OUTPUT_DIR)
print('Resolved model dir:', MODEL_DIR)

## 4. Post-training inference tests

The following cells load the fine-tuned model from the latest saved checkpoint and test it with multiple prompt styles, similar to the earlier Ollama test script.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
import re

ACTION_MAP = {
    0: 'do nothing',
    1: 'fire left orientation engine',
    2: 'fire main engine',
    3: 'fire right orientation engine',
}

TEST_CASES = [
    {
        'name': 'high_center',
        'state': '[x=0.0005, y=1.4126, vx=0.0492, vy=0.0739, angle=-0.0006, angular_vel=-0.0112, left_leg=0.0000, right_leg=0.0000]'
    },
    {
        'name': 'left_tilt',
        'state': '[x=-0.0500, y=0.4000, vx=-0.0200, vy=-0.1500, angle=0.3000, angular_vel=0.2500, left_leg=0.0000, right_leg=0.0000]'
    },
    {
        'name': 'right_tilt',
        'state': '[x=0.3000, y=0.9000, vx=0.3500, vy=-0.2500, angle=-0.2200, angular_vel=-0.1500, left_leg=0.0000, right_leg=0.0000]'
    },
    {
        'name': 'touchdown',
        'state': '[x=0.0000, y=0.0800, vx=0.0000, vy=-0.0200, angle=0.0000, angular_vel=0.0000, left_leg=1.0000, right_leg=1.0000]'
    },
    {
        'name': 'left_far',
        'state': '[x=-0.4200, y=0.6500, vx=-0.4800, vy=-0.2800, angle=0.3300, angular_vel=0.2400, left_leg=0.0000, right_leg=0.0000]'
    },
    {
        'name': 'descending_main',
        'state': '[x=0.0200, y=0.2500, vx=0.0100, vy=-0.0500, angle=0.0100, angular_vel=0.0000, left_leg=1.0000, right_leg=1.0000]'
    },
]

def build_simple_prompt(state: str) -> str:
    return f'State: {state}. What action should the lander take?'

def build_strict_prompt(state: str) -> str:
    return (
        'Return exactly one action in one line.\n'
        'Valid outputs:\n'
        'Action: 0 (do nothing).\n'
        'Action: 1 (fire left orientation engine).\n'
        'Action: 2 (fire main engine).\n'
        'Action: 3 (fire right orientation engine).\n\n'
        f'State: {state}. What action should the lander take?'
    )

def build_structured_prompt(state: str) -> str:
    return (
        'You are a LunarLander policy model.\n\n'
        'Action space:\n'
        '0 = do nothing\n'
        '1 = fire left orientation engine\n'
        '2 = fire main engine\n'
        '3 = fire right orientation engine\n\n'
        f'Current state: {state}\n\n'
        'Choose the best action and return exactly one line in this format:\n'
        'Action: <id> (<description>).'
    )

PROMPT_BUILDERS = {
    'simple': build_simple_prompt,
    'strict': build_strict_prompt,
    'structured': build_structured_prompt,
}

def parse_action(text: str):
    normalized = text.strip().lower()
    match = re.search(r'action\s*:\s*([0-3])', normalized)
    if match:
        return int(match.group(1))
    if 'fire main engine' in normalized:
        return 2
    if 'fire left orientation engine' in normalized or 'fire left engine' in normalized:
        return 1
    if 'fire right orientation engine' in normalized or 'fire right engine' in normalized:
        return 3
    if 'do nothing' in normalized:
        return 0
    return None

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

inference_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
inference_model.eval()

def generate_one(prompt: str, max_new_tokens: int = 32) -> str:
    messages = [{'role': 'user', 'content': prompt}]
    if getattr(tokenizer, 'chat_template', None):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = prompt

    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    with torch.inference_mode():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

quick_prompt = build_strict_prompt(TEST_CASES[0]['state'])
print('Quick prompt:')
print(quick_prompt)
print('\nQuick response:')
print(generate_one(quick_prompt))

In [ ]:
results = []

for style_name, builder in PROMPT_BUILDERS.items():
    for case in TEST_CASES:
        prompt = builder(case['state'])
        raw = generate_one(prompt)
        action_id = parse_action(raw) if raw else None
        results.append({
            'prompt_style': style_name,
            'case': case['name'],
            'state': case['state'],
            'raw_output': raw,
            'parsed_action': action_id,
            'parsed_label': ACTION_MAP.get(action_id),
        })

results_df = pd.DataFrame(results)
display(results_df[['prompt_style', 'case', 'parsed_action', 'parsed_label', 'raw_output']])

print('Detailed results')
print('-' * 100)
for item in results:
    print(f"[{item['prompt_style']}] {item['case']}")
    print(f"state        : {item['state']}")
    print(f"raw output   : {item['raw_output']!r}")
    print(f"parsed action: {item['parsed_action']} ({item['parsed_label']})")
    print('-' * 100)

## 5. Save and download the fine-tuned model

If training and inference look good, archive the latest saved model directory and download it from Colab.

In [ ]:
from pathlib import Path
import shutil

MODEL_DIR = resolve_model_dir(OUTPUT_DIR)
archive_base = Path('/content/qwen25_05b_lunarlander_full_ft')
zip_path = shutil.make_archive(str(archive_base), 'zip', root_dir=str(MODEL_DIR))

print('Archived model dir:', MODEL_DIR)
print('Zip file:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print('Automatic download is only available inside Colab.')
    print('Reason:', exc)